# Aula 20 — Estratégias para manipulação de dados
**ETL, ELT e arquitetura Medallion — prática com PySpark**

**Objetivo:** acompanhar a preparação de dados industriais nas camadas Bronze, Silver e Gold.

Cada ciclo contém uma demonstração completa e uma atividade curta. Ao final, há um desafio integrador.

**Observação:** as camadas são representadas por DataFrames Spark para que o notebook funcione sem permissões de criação de catálogos ou tabelas. Em uma arquitetura real, as camadas também seriam persistidas, por exemplo, em tabelas Delta. ETL/ELT, CAP e Medallion são contextualizados nas apresentações.

## Ciclo 0 — Preparação dos dados
A base simula lotes de uma indústria. Existem registros duplicados e inconsistentes intencionalmente. Não é necessário importar arquivos.

In [ ]:
dados = [
    (101, "Linha A", "Manhã", 150, 6, 340.0),
    (102, "Linha A", "Tarde", 130, 7, 300.0),
    (103, "Linha B", "Manhã", 120, 12, 310.0),
    (104, "Linha B", "Noite", 100, 15, 280.0),
    (105, "Linha C", "Manhã", 210, 4, 470.0),
    (106, "Linha C", "Noite", 160, 5, 380.0),
    (107, "Linha A", "Noite", 110, 9, 295.0),
    (108, "Linha B", "Tarde", 140, 10, 335.0),
    (109, "Linha C", "Tarde", 200, 7, 450.0),
    (110, "Linha A", "Manhã", 175, 8, 390.0),
    (111, "Linha B", "Noite", 95, 10, 265.0),
    (112, "Linha C", "Manhã", 185, 5, 415.0),
    (113, "Linha A", "Tarde", 0, 0, 100.0),
    (114, "Linha B", "Noite", 130, 145, 340.0),
    (115, None, "Manhã", 170, 12, 390.0),
    (116, "Linha C", "Noite", 190, -3, 480.0),
    (117, "Linha C", "Tarde", 170, 11, None),
    (101, "Linha A", "Manhã", 150, 6, 340.0)
]

colunas = """
id_lote INT, linha STRING, turno STRING,
unidades_produzidas INT, unidades_refugadas INT, energia_kwh DOUBLE
"""

bronze = spark.createDataFrame(dados, schema=colunas)

## Ciclo 1 — Bronze: conhecer os dados
**Demonstração:** inspecionar os registros brutos com `printSchema()`, `select()`, `limit()`, `count()` e `display()`.

In [ ]:
bronze.printSchema()
display(bronze.select("id_lote", "linha", "unidades_produzidas").limit(6))
print("Total de registros na Bronze:", bronze.count())

**Atividade 1 — Faça você:** selecione `id_lote`, `turno` e `unidades_refugadas`. Exiba apenas **6 registros** e informe a quantidade total de registros da Bronze. Utilize as mesmas operações da demonstração.

In [ ]:
# TODO — Atividade 1
# Selecione as três colunas, mostre 6 registros e informe o total.

## Ciclo 2 — Silver: limpar os dados
**Demonstração:** eliminar duplicidades por lote e remover registros com linha ausente, produção inválida, refugo inconsistente ou energia ausente. Em produção, registros descartados devem ser rastreados.

In [ ]:
from pyspark.sql.functions import col

silver = (
    bronze
    .dropDuplicates(["id_lote"])
    .filter(col("linha").isNotNull())
    .filter(col("unidades_produzidas") > 0)
    .filter(col("unidades_refugadas") >= 0)
    .filter(col("unidades_refugadas") <= col("unidades_produzidas"))
    .filter(col("energia_kwh").isNotNull())
)

print("Registros na Bronze:", bronze.count())
print("Registros na Silver:", silver.count())
display(silver.limit(6))

# Demonstração adicional: filtrar lotes de um turno
lotes_manha = silver.filter(col("turno") == "Manhã")
display(lotes_manha.limit(5))
print("Lotes da manhã:", lotes_manha.count())

**Atividade 2 — Faça você:** crie `lotes_noite` filtrando a `silver` para `turno == "Noite"`. Exiba até **5 registros** e conte os lotes. Repita o filtro demonstrado para o turno da manhã.

In [ ]:
# TODO — Atividade 2
# Crie lotes_noite com filter(), use display() e count().

## Ciclo 3 — Silver: enriquecer os dados
**Demonstração:** criar duas colunas derivadas com `withColumn()` e `round()`: unidades aprovadas e taxa de refugo do lote.

In [ ]:
from pyspark.sql.functions import round

silver_enriquecida = (
    silver
    .withColumn(
        "unidades_aprovadas",
        col("unidades_produzidas") - col("unidades_refugadas")
    )
    .withColumn(
        "taxa_refugo_pct",
        round(col("unidades_refugadas") / col("unidades_produzidas") * 100, 2)
    )
)

display(
    silver_enriquecida.select(
        "id_lote", "unidades_produzidas",
        "unidades_aprovadas", "taxa_refugo_pct"
    ).limit(8)
)

**Atividade 3 — Faça você:** a partir de `silver_enriquecida`, crie `silver_energia` com a coluna `energia_por_unidade`, calculada por `energia_kwh / unidades_produzidas`, arredondada para **2 casas decimais**. Exiba o identificador do lote e a nova coluna.

In [ ]:
# TODO — Atividade 3
# Crie a coluna energia_por_unidade com withColumn() e round().

## Ciclo 4 — Gold: produzir indicadores
**Demonstração:** agrupar por linha, somar produção e refugo, calcular a energia média e a taxa de refugo total. A taxa é calculada sobre as **somas**, não pela média simples das taxas dos lotes.

In [ ]:
from pyspark.sql.functions import sum as sum_, avg

gold_linhas = (
    silver_enriquecida
    .groupBy("linha")
    .agg(
        sum_("unidades_produzidas").alias("producao_total"),
        sum_("unidades_refugadas").alias("refugo_total"),
        round(avg("energia_kwh"), 2).alias("energia_media_kwh")
    )
    .withColumn(
        "taxa_refugo_pct",
        round(col("refugo_total") / col("producao_total") * 100, 2)
    )
    .orderBy(col("taxa_refugo_pct").desc())
)

display(gold_linhas)

**Atividade 4 — Faça você:** crie `gold_turnos` repetindo **as mesmas agregações**, mas agrupando por `turno` em vez de `linha`. Ordene pela maior taxa de refugo e exiba a tabela.

In [ ]:
# TODO — Atividade 4
# Crie gold_turnos: groupBy(), agg(), withColumn(), orderBy() e display().

## Desafio final — Qual combinação de linha e turno exige atenção?
A gerência industrial quer uma tabela consolidada por **linha e turno**.

A partir de `silver_enriquecida`, construa `gold_linha_turno` com:
1. `producao_total`: soma de `unidades_produzidas`;
2. `refugo_total`: soma de `unidades_refugadas`;
3. `energia_media_kwh`: média de `energia_kwh` (2 casas decimais);
4. `taxa_refugo_pct`: `refugo_total / producao_total * 100` (2 casas decimais).

Ordene por taxa de refugo decrescente e use `display()`.

**Interpretação:** qual combinação de linha e turno apresenta a maior taxa de refugo? Explique em uma frase por que esse grupo deve ser investigado.

**Utilize apenas as operações já demonstradas nos ciclos.**

In [ ]:
# TODO — Desafio final
# Crie gold_linha_turno e identifique o grupo com maior taxa de refugo.